# PheCode Feature Construction and Models

Builds date-aware pre-index PheCode features from the cohort saved by Notebook 01, then trains leakage-audited XGBoost models with and without smoking history. Predictions are generated out of fold using the shared five-fold assignments created by Notebook 02.

The final section performs a 12-month temporal diagnostic-blackout sensitivity analysis to evaluate whether performance is driven by EHR features recorded during recent diagnostic workup.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
RUN_DIR = DATA_DIR / "paper_reproduction"
PRED_DIR = RUN_DIR / "predictions"
FIG_DIR = RUN_DIR / "figures"
for directory in [RUN_DIR, PRED_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BASE_COHORT_PATH = DATA_DIR / "paper_cohort.parquet"
PHECODE_COHORT_PATH = DATA_DIR / "demo_baseline_paper_reproduction.parquet"
if not BASE_COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {BASE_COHORT_PATH}. Run 01_Cohort_Construction.ipynb first."
    )

demo = pd.read_parquet(BASE_COHORT_PATH).reset_index(drop=True)
demo["person_id"] = pd.to_numeric(demo["person_id"], errors="raise").astype("int64")
demo["has_lungcancer_3Y"] = pd.to_numeric(
    demo["has_lungcancer_3Y"], errors="raise"
).astype(int)
cohort = demo.copy()

print("Base cohort:", cohort.shape)
print("Cases:", int(cohort["has_lungcancer_3Y"].sum()))


## Build pre-index PheCode features

Map source ICD-9-CM and ICD-10-CM diagnoses recorded on or before each participant's Lifestyle Survey date to PheCodes. Save both date-aware features for blackout analyses and a date-free modeling cohort for downstream notebooks.


In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


PHECODE_MAP_CANDIDATES = [
    Path.home() / "repos" / "lungcancer" / "Phecode_map12_filtered.csv",
    Path.home() / "lungcancer" / "Phecode_map12_filtered.csv",
    Path("Phecode_map12_filtered.csv"),
    DATA_DIR / "Phecode_map12_filtered.csv",
]


def load_phecode_map(path):
    last_err = None
    for enc in ["utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc, engine="python", sep=None)
            cols = {c.strip().lower(): c for c in df.columns}
            need = ["icd", "flag", "icdstring", "phecode", "phecodestring", "phecodecategory"]
            missing = [k for k in need if k not in cols]
            if missing:
                raise ValueError(f"Missing columns: {missing}")
            df = df.rename(columns={cols[k]: k for k in need})
            df["icd"] = (
                df["icd"].astype("string").str.strip().str.upper().str.replace(".", "", regex=False)
            )
            df["flag"] = df["flag"].astype("string").str.strip()
            df["vocabulary_id"] = df["flag"].map({"9": "ICD9CM", "10": "ICD10CM"}).astype("string")
            df = df.dropna(subset=["icd", "vocabulary_id", "phecode"])
            for col in ["phecode", "phecodestring", "phecodecategory"]:
                df[col] = df[col].astype("string").str.strip()
            return df[
                ["vocabulary_id", "icd", "phecode", "phecodestring", "phecodecategory"]
            ].drop_duplicates(subset=["vocabulary_id", "icd", "phecode"])
        except Exception as e:
            last_err = e
    raise last_err


def rebuild_per_patient_phecodes_from_bigquery(person_ids):
    from google.cloud import bigquery

    map_path = first_existing(PHECODE_MAP_CANDIDATES)
    if map_path is None:
        raise FileNotFoundError(
            "Need Phecode_map12_filtered.csv in the notebook or configured data directory."
        )
    if "WORKSPACE_CDR" not in os.environ:
        raise RuntimeError("WORKSPACE_CDR is not set. Rebuild must run inside All of Us.")

    phemap = load_phecode_map(map_path)
    print(
        f"Mapping loaded: {len(phemap):,} rows; "
        f"{phemap['phecode'].nunique():,} unique PheCodes"
    )

    client = bigquery.Client()
    workspace_cdr = os.environ["WORKSPACE_CDR"]
    person_ids = pd.Series(person_ids).dropna().astype("int64").drop_duplicates().sort_values().tolist()
    demo_dates = globals().get("demo")
    use_survey_dates = (
        isinstance(demo_dates, pd.DataFrame)
        and {"person_id", "lifestyle_survey_date"}.issubset(demo_dates.columns)
    )

    def chunker(seq, size=1000):
        for i in range(0, len(seq), size):
            yield seq[i : i + size]

    dfs = []
    for i, chunk in enumerate(chunker(person_ids, size=1000)):
        if use_survey_dates:
            chunk_dates = (
                demo_dates.loc[demo_dates["person_id"].isin(chunk), ["person_id", "lifestyle_survey_date"]]
                .dropna()
                .drop_duplicates("person_id")
                .copy()
            )
            chunk_dates["lifestyle_survey_date"] = pd.to_datetime(
                chunk_dates["lifestyle_survey_date"], errors="coerce"
            ).dt.strftime("%Y-%m-%d")
            chunk_dates = chunk_dates.dropna(subset=["lifestyle_survey_date"])
            if chunk_dates.empty:
                continue
            structs = ",\n".join(
                "STRUCT("
                f"{int(row.person_id)} AS person_id, "
                f"TIMESTAMP('{row.lifestyle_survey_date}') AS lifestyle_survey_date"
                ")"
                for row in chunk_dates.itertuples(index=False)
            )
            sql = f"""
            WITH cohort AS (
              SELECT * FROM UNNEST([{structs}])
            )
            SELECT
              co.person_id,
              co.condition_start_datetime,
              c.vocabulary_id,
              REPLACE(c.concept_code, '.', '') AS icd_code
            FROM `{workspace_cdr}.condition_occurrence` AS co
            JOIN cohort
              ON cohort.person_id = co.person_id
            JOIN `{workspace_cdr}.concept` AS c
              ON c.concept_id = co.condition_source_concept_id
            WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
              AND co.condition_start_datetime <= cohort.lifestyle_survey_date
            """
        else:
            person_list_sql = ", ".join(str(x) for x in chunk)
            sql = f"""
            SELECT
              co.person_id,
              co.condition_start_datetime,
              c.vocabulary_id,
              REPLACE(c.concept_code, '.', '') AS icd_code
            FROM `{workspace_cdr}.condition_occurrence` AS co
            JOIN `{workspace_cdr}.concept` AS c
              ON c.concept_id = co.condition_source_concept_id
            WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
              AND co.person_id IN ({person_list_sql})
            """
        df_chunk = client.query(sql).result().to_dataframe(create_bqstorage_client=True)
        dfs.append(df_chunk)
        print(f"chunk {i + 1}: total rows so far {sum(len(d) for d in dfs):,}")

    cond = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"Condition rows: {len(cond):,}; patients: {cond['person_id'].nunique():,}")

    cond["icd_code"] = cond["icd_code"].astype("string").str.strip().str.upper()
    cond["vocabulary_id"] = cond["vocabulary_id"].astype("string").str.strip()

    joined = cond.merge(
        phemap,
        left_on=["vocabulary_id", "icd_code"],
        right_on=["vocabulary_id", "icd"],
        how="left",
    )
    cond_mapped = joined.dropna(subset=["phecode"]).copy()
    cond_mapped["condition_start_date"] = pd.to_datetime(
        cond_mapped["condition_start_datetime"], errors="coerce", utc=True
    ).dt.tz_convert(None)

    cond_deduped = (
        cond_mapped
        .dropna(subset=["condition_start_date"])
        .sort_values(["person_id", "phecodestring", "condition_start_date"])
        .drop_duplicates(["person_id", "phecodestring"], keep="first")
        .copy()
    )

    presence_wide = (
        cond_deduped.assign(value=np.int8(1))
        .set_index(["person_id", "phecodestring"])["value"]
        .unstack(fill_value=0)
    )
    date_wide = cond_deduped.set_index(["person_id", "phecodestring"])[
        "condition_start_date"
    ].unstack()
    date_wide.columns = [f"{col}_date" for col in date_wide.columns]

    cond_wide = presence_wide.join(date_wide, how="left").reset_index()
    out = DATA_DIR / "per_patient_with_phe_preindex_rebuilt.parquet"
    cond_wide.to_parquet(out, index=False, compression="snappy")
    print("Saved rebuilt PheCode file:", out, cond_wide.shape)
    return cond_wide



In [ ]:
cond_wide = rebuild_per_patient_phecodes_from_bigquery(demo["person_id"])
demo_phecode_dates = demo.merge(
    cond_wide,
    on="person_id",
    how="left",
    validate="one_to_one",
)

date_cols = [
    column
    for column in demo_phecode_dates.columns
    if column.endswith("_date")
    and column not in {
        "date_of_birth",
        "lifestyle_survey_date",
        "lungcancer_date",
        "first_ehr_date",
        "last_ehr_date",
    }
]
cohort = demo_phecode_dates.drop(columns=date_cols, errors="ignore").copy()
cohort.to_parquet(
    PHECODE_COHORT_PATH,
    index=False,
    compression="snappy",
)
demo = cohort.copy()

print("Date-aware PheCode cohort:", demo_phecode_dates.shape)
print("Saved modeling cohort:", PHECODE_COHORT_PATH, cohort.shape)
print("Cases:", int(cohort["has_lungcancer_3Y"].sum()))


In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLD_PATH = RUN_DIR / "cv_folds.parquet"

def get_or_create_folds(df, n_splits=5, seed=42):
    expected = df[["person_id", "has_lungcancer_3Y"]].copy()
    if FOLD_PATH.exists():
        folds = pd.read_parquet(FOLD_PATH)
        check = expected.merge(folds, on="person_id", how="left", validate="one_to_one")
        if check["fold"].isna().any():
            raise ValueError("Saved fold file does not match the current cohort.")
        return check[["person_id", "fold"]]

    y = expected["has_lungcancer_3Y"]
    fold = np.full(len(expected), -1, dtype=int)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold_id, (_, test_idx) in enumerate(splitter.split(expected, y)):
        fold[test_idx] = fold_id
    folds = pd.DataFrame({"person_id": expected["person_id"], "fold": fold})
    folds.to_parquet(FOLD_PATH, index=False)
    return folds

folds = get_or_create_folds(cohort)
cohort = cohort.merge(folds, on="person_id", how="left", validate="one_to_one")
print(cohort.groupby("fold")["has_lungcancer_3Y"].agg(["count", "sum"]))

In [ ]:
import re
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier


def load_phecode_map(path):
    last_error = None
    for encoding in ["utf-8-sig", "cp1252", "latin1"]:
        try:
            mapping = pd.read_csv(
                path,
                dtype=str,
                encoding=encoding,
                engine="python",
                sep=None,
            )
            normalized = {str(c).strip().lower(): c for c in mapping.columns}
            if "phecodestring" not in normalized:
                raise ValueError("Phecode map lacks a phecodestring column")
            return mapping[normalized["phecodestring"]].astype("string").str.strip()
        except Exception as error:
            last_error = error
    raise last_error


map_candidates = [
    Path.home() / "repos" / "lungcancer" / "Phecode_map12_filtered.csv",
    Path.home() / "lungcancer" / "Phecode_map12_filtered.csv",
    Path("Phecode_map12_filtered.csv"),
    DATA_DIR / "Phecode_map12_filtered.csv",
]
map_path = next((path for path in map_candidates if path.exists()), None)
if map_path is None:
    raise FileNotFoundError("Could not locate Phecode_map12_filtered.csv")

phecode_names = set(load_phecode_map(map_path).dropna().tolist())
phecode_cols = [column for column in cohort.columns if column in phecode_names]

# Remove lung-cancer/bronchus malignancy features to avoid direct outcome leakage.
lung_pattern = re.compile(
    r"(lung|bronchus).*(cancer|carcinoma|malignan|neoplasm)|"
    r"(cancer|carcinoma|malignan|neoplasm).*(lung|bronchus)",
    re.I,
)
leakage_cols = [column for column in phecode_cols if lung_pattern.search(str(column))]
phecode_cols = [column for column in phecode_cols if column not in leakage_cols]

if len(phecode_cols) < 100:
    raise RuntimeError(
        f"Only {len(phecode_cols)} Phecode columns were identified. "
        "Check the mapping file and reconstructed cohort."
    )

phecode_frame = cohort[phecode_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
phecode_frame = phecode_frame.loc[:, phecode_frame.nunique(dropna=False).gt(1)]

demographic_columns = [
    column for column in [
        "age_at_survey_int",
        "sex_at_birth",
        "race_ethnicity",
        "education_group",
    ]
    if column in cohort.columns
]
if "age_at_survey_int" not in demographic_columns:
    raise RuntimeError("age_at_survey_int is required for the demographic model")

demographic_frame = cohort[demographic_columns].copy()
demographic_frame["age_at_survey_int"] = pd.to_numeric(
    demographic_frame["age_at_survey_int"], errors="coerce"
)
categorical_demographics = [
    column for column in demographic_columns if column != "age_at_survey_int"
]
demographic_frame = pd.get_dummies(
    demographic_frame,
    columns=categorical_demographics,
    dummy_na=True,
)
demographic_frame = demographic_frame.apply(pd.to_numeric, errors="coerce")
demographic_frame["age_at_survey_int"] = demographic_frame[
    "age_at_survey_int"
].fillna(demographic_frame["age_at_survey_int"].median())

X = pd.concat([demographic_frame, phecode_frame], axis=1).fillna(0)

# XGBoost prohibits [, ], and < in feature names. Preserve a lookup to the
# original demographic/Phecode labels for interpretation while fitting with
# safe names.
original_feature_names = list(X.columns)
safe_feature_names = [
    re.sub(
        r"[^0-9A-Za-z_(),]+",
        "_",
        str(column).replace("[", "(").replace("]", ")").replace("<", "lt"),
    )
    for column in original_feature_names
]
X.columns = safe_feature_names
X = X.loc[:, ~X.columns.duplicated()]
feature_name_lookup = dict(zip(safe_feature_names, original_feature_names))
y = cohort["has_lungcancer_3Y"].astype(int)

print("Phecode mapping:", map_path)
print("Demographic source features:", demographic_columns)
print("Encoded demographic features:", demographic_frame.shape[1])
print("Phecode features:", phecode_frame.shape[1])
print("Total demographic + Phecode features:", X.shape[1])
print("Excluded possible leakage features:", leakage_cols)
print("Smoking/lifestyle predictors included: 0")

oof = np.zeros(len(cohort), dtype=float)
importance = []
fold_aurocs = []
fold_auprcs = []

for fold_id in sorted(cohort["fold"].unique()):
    test = cohort["fold"].eq(fold_id).to_numpy()
    train = ~test

    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X.loc[train], y.loc[train])
    oof[test] = model.predict_proba(X.loc[test])[:, 1]

    fold_auc = roc_auc_score(y.loc[test], oof[test])
    fold_ap = average_precision_score(y.loc[test], oof[test])
    fold_aurocs.append(fold_auc)
    fold_auprcs.append(fold_ap)
    importance.append(
        pd.Series(
            model.get_booster().get_score(importance_type="gain"),
            name=f"fold_{fold_id}",
        )
    )
    print(f"Fold {fold_id + 1}: AUROC={fold_auc:.4f}; AUPRC={fold_ap:.4f}")

pred = pd.DataFrame({
    "person_id": cohort["person_id"],
    "y_true": y,
    "prob": oof,
    "model": "Demographic_PheCode",
})
pred.to_parquet(PRED_DIR / "Demographic_PheCode_predictions.parquet", index=False)

# Remove the legacy output from the earlier notebook version, which combined
# clinical/lifestyle variables with Phecodes despite being labeled primary.
legacy_prediction = PRED_DIR / "Hybrid_PheCode_predictions.parquet"
if legacy_prediction.exists():
    legacy_prediction.unlink()
    print("Removed stale legacy prediction file:", legacy_prediction)
ehr_only_prediction = PRED_DIR / "EHR_only_PheCode_predictions.parquet"
if ehr_only_prediction.exists():
    ehr_only_prediction.unlink()
    print("Removed superseded EHR-only prediction file:", ehr_only_prediction)

fi = (
    pd.concat(importance, axis=1)
    .fillna(0)
    .mean(axis=1)
    .sort_values(ascending=False)
    .rename("mean_gain")
    .reset_index()
    .rename(columns={"index": "feature"})
)
fi["original_feature_label"] = fi["feature"].map(feature_name_lookup).fillna(fi["feature"])
fi.to_csv(RUN_DIR / "Demographic_PheCode_feature_importance.csv", index=False)

print("OOF AUROC:", roc_auc_score(y, oof))
print("OOF AUPRC:", average_precision_score(y, oof))
print("OOF Brier:", brier_score_loss(y, oof))
print(f"Fold AUROC: {np.mean(fold_aurocs):.3f} ± {np.std(fold_aurocs):.3f}")
print(f"Fold AUPRC: {np.mean(fold_auprcs):.3f} ± {np.std(fold_auprcs):.3f}")
display(fi.head(30))


## Clinician-facing fixed-capacity performance

Participants are ranked by out-of-fold predicted risk. Sensitivity and PPV are reported after selecting the top 5%, 10%, and 20% of the cohort.


In [ ]:
def top_capacity_metrics(predictions, fractions=(0.05, 0.10, 0.20)):
    ranked = predictions.sort_values(
        ["prob", "person_id"], ascending=[False, True]
    ).reset_index(drop=True)
    y_values = ranked["y_true"].astype(int).to_numpy()
    total_cases = int(y_values.sum())
    rows = []

    for fraction in fractions:
        selected_n = int(np.floor(len(ranked) * fraction))
        selected = np.zeros(len(ranked), dtype=bool)
        selected[:selected_n] = True

        tp = int((selected & (y_values == 1)).sum())
        fp = int((selected & (y_values == 0)).sum())
        fn = int((~selected & (y_values == 1)).sum())
        tn = int((~selected & (y_values == 0)).sum())

        rows.append({
            "Capacity": f"Top {int(fraction * 100)}%",
            "Selected": selected_n,
            "Cases captured": tp,
            "Sensitivity": tp / (tp + fn),
            "Specificity": tn / (tn + fp),
            "PPV": tp / (tp + fp),
            "Screened per observed case": selected_n / tp if tp else np.inf,
        })

    return pd.DataFrame(rows).set_index("Capacity")


ehr_capacity_metrics = top_capacity_metrics(pred)
display(ehr_capacity_metrics.style.format({
    "Sensitivity": "{:.1%}",
    "Specificity": "{:.1%}",
    "PPV": "{:.1%}",
    "Screened per observed case": "{:.1f}",
}))

ehr_summary = pd.DataFrame({
    "AUROC": [roc_auc_score(pred["y_true"], pred["prob"])],
    "AUPRC": [average_precision_score(pred["y_true"], pred["prob"])],
    "Top 5% Sens./PPV": [
        f"{ehr_capacity_metrics.loc['Top 5%', 'Sensitivity']:.1%} / "
        f"{ehr_capacity_metrics.loc['Top 5%', 'PPV']:.1%}"
    ],
    "Top 10% Sens./PPV": [
        f"{ehr_capacity_metrics.loc['Top 10%', 'Sensitivity']:.1%} / "
        f"{ehr_capacity_metrics.loc['Top 10%', 'PPV']:.1%}"
    ],
    "Top 20% Sens./PPV": [
        f"{ehr_capacity_metrics.loc['Top 20%', 'Sensitivity']:.1%} / "
        f"{ehr_capacity_metrics.loc['Top 20%', 'PPV']:.1%}"
    ],
}, index=["Demographic + Phecode"])

display(ehr_summary.style.format({"AUROC": "{:.3f}", "AUPRC": "{:.3f}"}))
ehr_summary.to_csv(RUN_DIR / "Demographic_PheCode_slide_metrics.csv")


## Demographic + Phecode + smoking model

This secondary hybrid model adds the four smoking constructs used in the published table: smoking status, usual cigarettes/day, smoking duration, and years since quitting. Missing numeric smoking values are imputed within each training fold; missing smoking status is retained as an explicit category. The model uses the same cohort and saved outer folds as the demographic + Phecode model.

In [ ]:
from sklearn.impute import SimpleImputer


def cohort_numeric(name):
    if name not in cohort.columns:
        return pd.Series(np.nan, index=cohort.index, dtype=float, name=name)
    return pd.to_numeric(cohort[name], errors="coerce").astype(float)


smoking_status = (
    cohort["smoking_status"].astype("string").str.strip().str.lower()
)
smoking_status = smoking_status.where(
    smoking_status.isin(["current", "former"]), "missing"
)

# Usual/lifetime cigarettes per day is appropriate for both current and
# former smokers. Current daily cigarettes is only a fallback for current
# smokers; zeros are treated as missing.
usual_cigarettes = cohort_numeric("Smoking: Average Daily Cigarette Number")
usual_cigarettes = usual_cigarettes.mask(usual_cigarettes <= 0)
current_cigarettes = cohort_numeric("Smoking: Current Daily Cigarette Number")
current_cigarettes = current_cigarettes.mask(current_cigarettes <= 0)
current_fallback = smoking_status.eq("current") & usual_cigarettes.isna()
usual_cigarettes.loc[current_fallback] = current_cigarettes.loc[current_fallback]

smoking_duration = cohort_numeric("Smoking: Number Of Years").mask(lambda s: s < 0)
years_since_quit = cohort_numeric("duration_of_quitting").mask(lambda s: s < 0)
years_since_quit.loc[smoking_status.eq("current")] = 0.0

smoking_frame = pd.DataFrame({
    "smoking_cigarettes_per_day": usual_cigarettes,
    "smoking_duration_years": smoking_duration,
    "smoking_years_since_quit": years_since_quit,
}, index=cohort.index)
smoking_frame = pd.concat([
    smoking_frame,
    pd.get_dummies(
        smoking_status, prefix="smoking_status", dtype=float
    ),
], axis=1)

X_smoking = pd.concat([X.copy(), smoking_frame], axis=1)
X_smoking.columns = [
    re.sub(
        r"[^0-9A-Za-z_(),]+", "_",
        str(column).replace("[", "(").replace("]", ")").replace("<", "lt"),
    )
    for column in X_smoking.columns
]
X_smoking = X_smoking.loc[:, ~X_smoking.columns.duplicated()]

print("Smoking source variables: status, usual cigarettes/day, duration, quit time")
print("Smoking-status counts:")
display(smoking_status.value_counts(dropna=False).rename("n").to_frame())
print("Total demographic + Phecode + smoking features:", X_smoking.shape[1])
print("Missing cigarettes/day:", int(usual_cigarettes.isna().sum()))
print("Missing smoking duration:", int(smoking_duration.isna().sum()))
print("Missing years since quitting:", int(years_since_quit.isna().sum()))

hybrid_oof = np.zeros(len(cohort), dtype=float)
hybrid_fold_aurocs = []
hybrid_fold_auprcs = []

for fold_id in sorted(cohort["fold"].unique()):
    test = cohort["fold"].eq(fold_id).to_numpy()
    train = ~test

    # Fit imputation using training participants only.
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_smoking.loc[train])
    X_test = imputer.transform(X_smoking.loc[test])

    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y.loc[train])
    hybrid_oof[test] = model.predict_proba(X_test)[:, 1]

    fold_auc = roc_auc_score(y.loc[test], hybrid_oof[test])
    fold_ap = average_precision_score(y.loc[test], hybrid_oof[test])
    hybrid_fold_aurocs.append(fold_auc)
    hybrid_fold_auprcs.append(fold_ap)
    print(f"Fold {fold_id + 1}: AUROC={fold_auc:.4f}; AUPRC={fold_ap:.4f}")

hybrid_pred = pd.DataFrame({
    "person_id": cohort["person_id"],
    "y_true": y,
    "prob": hybrid_oof,
    "model": "Demographic_PheCode_Smoking",
})
hybrid_path = PRED_DIR / "Demographic_PheCode_Smoking_predictions.parquet"
hybrid_pred.to_parquet(hybrid_path, index=False)

print("Saved:", hybrid_path)
print("OOF AUROC:", roc_auc_score(y, hybrid_oof))
print("OOF AUPRC:", average_precision_score(y, hybrid_oof))
print("OOF Brier:", brier_score_loss(y, hybrid_oof))
print(f"Fold AUROC: {np.mean(hybrid_fold_aurocs):.3f} ± {np.std(hybrid_fold_aurocs):.3f}")
print(f"Fold AUPRC: {np.mean(hybrid_fold_auprcs):.3f} ± {np.std(hybrid_fold_auprcs):.3f}")


In [ ]:
def slide_metric_row(label, predictions):
    capacity = top_capacity_metrics(predictions)
    return {
        "Model": label,
        "AUROC": roc_auc_score(predictions["y_true"], predictions["prob"]),
        "AUPRC": average_precision_score(predictions["y_true"], predictions["prob"]),
        "Top 5% Sens./PPV": (
            f"{capacity.loc['Top 5%', 'Sensitivity']:.1%} / "
            f"{capacity.loc['Top 5%', 'PPV']:.1%}"
        ),
        "Top 10% Sens./PPV": (
            f"{capacity.loc['Top 10%', 'Sensitivity']:.1%} / "
            f"{capacity.loc['Top 10%', 'PPV']:.1%}"
        ),
        "Top 20% Sens./PPV": (
            f"{capacity.loc['Top 20%', 'Sensitivity']:.1%} / "
            f"{capacity.loc['Top 20%', 'PPV']:.1%}"
        ),
    }


comparison_summary = pd.DataFrame([
    slide_metric_row("Demographic + Phecode", pred),
    slide_metric_row("Demographic + Phecode + Smoking", hybrid_pred),
]).set_index("Model")
display(comparison_summary.style.format({"AUROC": "{:.3f}", "AUPRC": "{:.3f}"}))
comparison_summary.to_csv(
    RUN_DIR / "Demographic_PheCode_Smoking_slide_metrics.csv"
)



---



# Part II: Temporal Diagnostic-Blackout Sensitivity Analysis

This section reruns the hybrid PheCode model after excluding diagnosis-derived features recorded within 12 months before the baseline Lifestyle Survey date. It tests whether recent EHR signals reflect diagnostic workup rather than antecedent risk.


In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

datadir = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
datadir.mkdir(parents=True, exist_ok=True)
RUN_DIR = datadir / "paper_reproduction"
COHORT_PATH = datadir / "demo_baseline_paper_reproduction.parquet"
FOLD_PATH = RUN_DIR / "cv_folds.parquet"

BLACKOUT_WINDOWS_DAYS = [365]
N_SPLITS = 5
RANDOM_STATE = 42
RUN_EHR_ONLY = False

PHECODE_MAP_CANDIDATES = [
    Path("Phecode_map12_filtered.csv"),
    datadir / "Phecode_map12_filtered.csv",
]

print("Data directory:", datadir)



## Helpers



In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None


def to_datetime_naive(s):
    return pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert(None)


def norm_text(s):
    if pd.isna(s):
        return ""
    return re.sub(r"\s+", " ", str(s).strip().lower())


def add_race_ethnicity(df):
    if "race_ethnicity" in df.columns:
        df["race_ethnicity"] = df["race_ethnicity"].fillna("Missing")
        return df
    if not {"race", "ethnicity"}.issubset(df.columns):
        df["race_ethnicity"] = "Missing"
        return df

    race_missing = {"none indicated", "i prefer not to answer", "pmi: skip", "none of these", ""}
    eth_missing = {
        "no matching concept",
        "pmi: prefer not to answer",
        "pmi: skip",
        "what race ethnicity: race ethnicity none of these",
        "",
    }

    def make(row):
        race = norm_text(row.get("race"))
        eth = norm_text(row.get("ethnicity"))
        if eth == "hispanic or latino":
            return "Hispanic"
        if eth in eth_missing:
            return "Missing"
        if eth == "not hispanic or latino":
            if race == "white":
                return "White"
            if race == "black or african american":
                return "Black"
            if race == "asian":
                return "Asian"
            if race == "american indian or alaska native":
                return "American Indian or Alaskan Native"
            if race == "native hawaiian or other pacific islander":
                return "Native Hawaiian or Pacific Islander"
            if race in race_missing:
                return "Missing"
        return "Missing"

    df["race_ethnicity"] = df.apply(make, axis=1)
    return df


def add_age_at_survey(df):
    if "age_at_survey_int" in df.columns:
        return df
    if {"date_of_birth", "lifestyle_survey_date"}.issubset(df.columns):
        dob = to_datetime_naive(df["date_of_birth"])
        survey = to_datetime_naive(df["lifestyle_survey_date"])
        df["age_at_survey"] = (survey - dob).dt.days / 365.25
        df["age_at_survey_int"] = df["age_at_survey"].round(0).astype("Int64")
    return df


def add_3y_outcome(df):
    if "has_lungcancer_3Y" in df.columns:
        df["has_lungcancer_3Y"] = df["has_lungcancer_3Y"].astype(int)
        return df
    required = {"lungcancer_date", "lifestyle_survey_date"}
    if not required.issubset(df.columns):
        raise ValueError("Need has_lungcancer_3Y or lungcancer_date + lifestyle_survey_date.")
    survey = to_datetime_naive(df["lifestyle_survey_date"])
    lung = to_datetime_naive(df["lungcancer_date"])
    df["has_lungcancer_3Y"] = (
        lung.notna()
        & (lung >= survey)
        & (lung <= survey + pd.DateOffset(years=3))
    ).astype(int)
    return df


def remove_prior_lung_cancer(df):
    if not {"lungcancer_date", "lifestyle_survey_date"}.issubset(df.columns):
        return df
    survey = to_datetime_naive(df["lifestyle_survey_date"])
    lung = to_datetime_naive(df["lungcancer_date"])
    prior = lung.notna() & (lung < survey)
    print("Removing prior lung cancer rows:", int(prior.sum()))
    return df.loc[~prior].copy()



## Load Base Cohort



In [ ]:
if not COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {COHORT_PATH}. Run "
        "01_Cohort_Construction.ipynb first. "
        "this sensitivity-analysis section must use the same cohort artifact as Notebook 02."
    )

demo = pd.read_parquet(COHORT_PATH).reset_index(drop=True)
demo["person_id"] = pd.to_numeric(demo["person_id"], errors="raise").astype("int64")
print("Loaded paper-reproduction cohort:", COHORT_PATH, demo.shape)

demo["has_lungcancer_3Y"] = pd.to_numeric(
    demo["has_lungcancer_3Y"], errors="raise"
).astype(int)
demo = add_age_at_survey(demo)
demo = add_race_ethnicity(demo)
demo["lifestyle_survey_date"] = to_datetime_naive(demo["lifestyle_survey_date"])

print("Final demo:", demo.shape, "cases:", int(demo["has_lungcancer_3Y"].sum()))



## Load Or Rebuild Date-Aware PheCodes



In [ ]:
def load_phecode_map(path):
    last_err = None
    for enc in ["utf-8-sig", "cp1252", "latin1"]:
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc, engine="python", sep=None)
            cols = {c.strip().lower(): c for c in df.columns}
            need = ["icd", "flag", "icdstring", "phecode", "phecodestring", "phecodecategory"]
            missing = [k for k in need if k not in cols]
            if missing:
                raise ValueError(f"Missing columns: {missing}")
            df = df.rename(columns={cols[k]: k for k in need})
            df["icd"] = (
                df["icd"].astype("string").str.strip().str.upper().str.replace(".", "", regex=False)
            )
            df["flag"] = df["flag"].astype("string").str.strip()
            df["vocabulary_id"] = df["flag"].map({"9": "ICD9CM", "10": "ICD10CM"}).astype("string")
            df = df.dropna(subset=["icd", "vocabulary_id", "phecode"])
            for col in ["phecode", "phecodestring", "phecodecategory"]:
                df[col] = df[col].astype("string").str.strip()
            return df[
                ["vocabulary_id", "icd", "phecode", "phecodestring", "phecodecategory"]
            ].drop_duplicates(subset=["vocabulary_id", "icd", "phecode"])
        except Exception as e:
            last_err = e
    raise last_err


def rebuild_per_patient_phecodes_from_bigquery(person_ids):
    from google.cloud import bigquery

    map_path = first_existing(PHECODE_MAP_CANDIDATES)
    if map_path is None:
        raise FileNotFoundError(
            "Need Phecode_map12_filtered.csv in the notebook or configured data directory."
        )
    if "WORKSPACE_CDR" not in os.environ:
        raise RuntimeError("WORKSPACE_CDR is not set. Rebuild must run inside All of Us.")

    phemap = load_phecode_map(map_path)
    print(
        f"Mapping loaded: {len(phemap):,} rows; "
        f"{phemap['phecode'].nunique():,} unique PheCodes"
    )

    client = bigquery.Client()
    workspace_cdr = os.environ["WORKSPACE_CDR"]
    person_ids = pd.Series(person_ids).dropna().astype("int64").drop_duplicates().sort_values().tolist()
    demo_dates = globals().get("demo")
    use_survey_dates = (
        isinstance(demo_dates, pd.DataFrame)
        and {"person_id", "lifestyle_survey_date"}.issubset(demo_dates.columns)
    )

    def chunker(seq, size=1000):
        for i in range(0, len(seq), size):
            yield seq[i : i + size]

    dfs = []
    for i, chunk in enumerate(chunker(person_ids, size=1000)):
        if use_survey_dates:
            chunk_dates = (
                demo_dates.loc[demo_dates["person_id"].isin(chunk), ["person_id", "lifestyle_survey_date"]]
                .dropna()
                .drop_duplicates("person_id")
                .copy()
            )
            chunk_dates["lifestyle_survey_date"] = pd.to_datetime(
                chunk_dates["lifestyle_survey_date"], errors="coerce"
            ).dt.strftime("%Y-%m-%d")
            chunk_dates = chunk_dates.dropna(subset=["lifestyle_survey_date"])
            if chunk_dates.empty:
                continue
            structs = ",\n".join(
                "STRUCT("
                f"{int(row.person_id)} AS person_id, "
                f"TIMESTAMP('{row.lifestyle_survey_date}') AS lifestyle_survey_date"
                ")"
                for row in chunk_dates.itertuples(index=False)
            )
            sql = f"""
            WITH cohort AS (
              SELECT * FROM UNNEST([{structs}])
            )
            SELECT
              co.person_id,
              co.condition_start_datetime,
              c.vocabulary_id,
              REPLACE(c.concept_code, '.', '') AS icd_code
            FROM `{workspace_cdr}.condition_occurrence` AS co
            JOIN cohort
              ON cohort.person_id = co.person_id
            JOIN `{workspace_cdr}.concept` AS c
              ON c.concept_id = co.condition_source_concept_id
            WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
              AND co.condition_start_datetime <= cohort.lifestyle_survey_date
            """
        else:
            person_list_sql = ", ".join(str(x) for x in chunk)
            sql = f"""
            SELECT
              co.person_id,
              co.condition_start_datetime,
              c.vocabulary_id,
              REPLACE(c.concept_code, '.', '') AS icd_code
            FROM `{workspace_cdr}.condition_occurrence` AS co
            JOIN `{workspace_cdr}.concept` AS c
              ON c.concept_id = co.condition_source_concept_id
            WHERE c.vocabulary_id IN ('ICD9CM', 'ICD10CM')
              AND co.person_id IN ({person_list_sql})
            """
        df_chunk = client.query(sql).result().to_dataframe(create_bqstorage_client=True)
        dfs.append(df_chunk)
        print(f"chunk {i + 1}: total rows so far {sum(len(d) for d in dfs):,}")

    cond = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"Condition rows: {len(cond):,}; patients: {cond['person_id'].nunique():,}")

    cond["icd_code"] = cond["icd_code"].astype("string").str.strip().str.upper()
    cond["vocabulary_id"] = cond["vocabulary_id"].astype("string").str.strip()

    joined = cond.merge(
        phemap,
        left_on=["vocabulary_id", "icd_code"],
        right_on=["vocabulary_id", "icd"],
        how="left",
    )
    cond_mapped = joined.dropna(subset=["phecode"]).copy()
    cond_mapped["condition_start_date"] = pd.to_datetime(
        cond_mapped["condition_start_datetime"], errors="coerce", utc=True
    ).dt.tz_convert(None)

    cond_deduped = (
        cond_mapped
        .dropna(subset=["condition_start_date"])
        .sort_values(["person_id", "phecodestring", "condition_start_date"])
        .drop_duplicates(["person_id", "phecodestring"], keep="first")
        .copy()
    )

    presence_wide = (
        cond_deduped.assign(value=np.int8(1))
        .set_index(["person_id", "phecodestring"])["value"]
        .unstack(fill_value=0)
    )
    date_wide = cond_deduped.set_index(["person_id", "phecodestring"])[
        "condition_start_date"
    ].unstack()
    date_wide.columns = [f"{col}_date" for col in date_wide.columns]

    cond_wide = presence_wide.join(date_wide, how="left").reset_index()
    out = datadir / "per_patient_with_phe_preindex_rebuilt.parquet"
    cond_wide.to_parquet(out, index=False, compression="snappy")
    print("Saved rebuilt PheCode file:", out, cond_wide.shape)
    return cond_wide



In [ ]:
date_aware_phe_path = first_existing([
    datadir / "per_patient_with_phe_preindex_rebuilt.parquet",
    datadir / "demo_phecode_dates.parquet",
])

date_aware_phe = None
if date_aware_phe_path is not None:
    candidate = pd.read_parquet(date_aware_phe_path)
    candidate_date_cols = [
        c for c in candidate.columns
        if c.endswith("_date")
        and c not in {"date_of_birth", "lifestyle_survey_date", "lungcancer_date"}
        and c[:-5] in candidate.columns
    ]
    if candidate_date_cols:
        date_aware_phe = candidate
        print(
            "Loaded date-aware PheCode file:",
            date_aware_phe_path,
            candidate.shape,
            f"({len(candidate_date_cols):,} date columns)",
        )
    else:
        print(
            "Saved PheCode file has no condition-date columns and cannot support a blackout; "
            "rebuilding it from BigQuery:",
            date_aware_phe_path,
        )

if date_aware_phe is None:
    print("No saved date-aware PheCode matrix found. Rebuilding from BigQuery...")
    date_aware_phe = rebuild_per_patient_phecodes_from_bigquery(demo["person_id"])

paired_date_cols = [
    c for c in date_aware_phe.columns
    if c.endswith("_date")
    and c not in {"date_of_birth", "lifestyle_survey_date", "lungcancer_date"}
    and c[:-5] in date_aware_phe.columns
]
if not paired_date_cols:
    raise RuntimeError(
        "The date-aware PheCode file has no matched indicator/date pairs. "
        "Rerun Notebook 01 through the date-aware PheCode rebuild cell."
    )
paired_flag_cols = [c[:-5] for c in paired_date_cols]
condition_matrix = date_aware_phe[
    ["person_id", *paired_flag_cols, *paired_date_cols]
].copy()

# Notebook 02's cohort artifact already contains PheCode indicators. Drop those
# columns before attaching the date-aware versions to avoid _x/_y duplicates.
demo_without_phecodes = demo.drop(
    columns=[*paired_flag_cols, *paired_date_cols],
    errors="ignore",
)
demo_phecode_dates = demo_without_phecodes.merge(
    condition_matrix,
    on="person_id",
    how="left",
    validate="one_to_one",
)

refresh_cols = [c for c in demo.columns if c in {
    "person_id", "lifestyle_survey_date", "lungcancer_date", "has_lungcancer",
    "has_lungcancer_post", "has_lungcancer_12M", "has_lungcancer_24M",
    "has_lungcancer_3Y", "has_lungcancer_6Y", "age_at_survey", "age_at_survey_int",
    "race_ethnicity", "sex_at_birth", "bmi", "education_group", "smoking_status",
    "duration_of_quitting", "Smoking: Number Of Years",
    "Smoking: Current Daily Cigarette Number", "COPD_pre", "family_lung",
}]
stale = [c for c in refresh_cols if c != "person_id" and c in demo_phecode_dates.columns]
demo_phecode_dates = demo_phecode_dates.drop(columns=stale, errors="ignore").merge(
    demo[refresh_cols], on="person_id", how="inner", validate="one_to_one"
)
if len(demo_phecode_dates) != len(demo):
    raise ValueError(
        f"Date-aware PheCode data cover {len(demo_phecode_dates):,} of "
        f"{len(demo):,} Notebook 02 participants. Rebuild the date-aware PheCode file."
    )

blackout_date_cols = [
    c for c in demo_phecode_dates.columns
    if c.endswith("_date")
    and c not in {"date_of_birth", "lifestyle_survey_date", "lungcancer_date"}
    and c[:-5] in demo_phecode_dates.columns
]
if not blackout_date_cols:
    raise RuntimeError(
        "No paired PheCode indicator/date columns are available for the blackout analysis. "
        "Rerun 01_Cohort_Construction.ipynb through the date-aware PheCode rebuild cell."
    )
print(f"Blackout-ready PheCode features: {len(blackout_date_cols):,}")

demo_phecode_dates["lifestyle_survey_date"] = to_datetime_naive(
    demo_phecode_dates["lifestyle_survey_date"]
)
demo_phecode_dates = add_age_at_survey(demo_phecode_dates)
demo_phecode_dates = add_race_ethnicity(demo_phecode_dates)

print("Ready for temporal sensitivity:", demo_phecode_dates.shape)
print("Cases:", int(demo_phecode_dates["has_lungcancer_3Y"].sum()))

if not FOLD_PATH.exists():
    raise FileNotFoundError(
        f"Missing {FOLD_PATH}. Run 02_Clinical_Risk_Score_Benchmarks.ipynb "
        "to create the shared five-fold assignments."
    )
folds = pd.read_parquet(FOLD_PATH)
folds["person_id"] = pd.to_numeric(folds["person_id"], errors="raise").astype("int64")
demo_phecode_dates = demo_phecode_dates.drop(columns=["fold"], errors="ignore").merge(
    folds[["person_id", "fold"]],
    on="person_id",
    how="left",
    validate="one_to_one",
)
if demo_phecode_dates["fold"].isna().any():
    raise ValueError("Saved Notebook 02 folds do not match the temporal-analysis cohort.")
demo_phecode_dates["fold"] = demo_phecode_dates["fold"].astype(int)
print("Using Notebook 02 fold assignments")
print(demo_phecode_dates.groupby("fold")["has_lungcancer_3Y"].agg(["count", "sum"]))



## Feature Matrix Builders



In [ ]:
def condition_date_columns(df):
    return [
        c
        for c in df.columns
        if c.endswith("_date") and c not in {"date_of_birth", "lifestyle_survey_date", "lungcancer_date"}
    ]


def apply_temporal_blackout(df, blackout_days):
    out = df.copy()
    survey = to_datetime_naive(out["lifestyle_survey_date"])
    cutoff = survey - pd.to_timedelta(blackout_days, unit="D")

    flips = []
    date_cols = condition_date_columns(out)
    for date_col in date_cols:
        flag_col = date_col[:-5]
        if flag_col not in out.columns:
            continue
        cond_date = to_datetime_naive(out[date_col])
        flag_vals = pd.to_numeric(out[flag_col], errors="coerce").fillna(0).astype("int8")
        remove_mask = cond_date.notna() & (cond_date > cutoff)
        flips.append({
            "condition": flag_col,
            "n_removed": int((remove_mask & (flag_vals == 1)).sum()),
        })
        out.loc[remove_mask, flag_col] = np.int8(0)
        out[flag_col] = pd.to_numeric(out[flag_col], errors="coerce").fillna(0).astype("int8")

    if not flips:
        raise RuntimeError(
            "No paired PheCode indicator/date columns were found. "
            "Rebuild per_patient_with_phe_preindex_rebuilt.parquet in Notebook 01."
        )
    flips_df = (
        pd.DataFrame(flips, columns=["condition", "n_removed"])
        .sort_values("n_removed", ascending=False)
        .reset_index(drop=True)
    )
    out = out.drop(columns=date_cols, errors="ignore")
    return out, flips_df


def add_or_refresh_cancer_history(df):
    if "cancer_history" in df.columns:
        df["cancer_history"] = pd.to_numeric(df["cancer_history"], errors="coerce").fillna(0).astype("int8")
        return df
    cancer_terms = [
        "cancer", "carcinoma", "malignant", "neoplasm", "tumor",
        "tumour", "sarcoma", "leukemia", "lymphoma", "melanoma",
    ]
    pattern = re.compile("|".join(cancer_terms), re.IGNORECASE)
    ignore = {
        "has_lungcancer", "has_lungcancer_post", "has_lungcancer_12M",
        "has_lungcancer_24M", "has_lungcancer_3Y", "has_lungcancer_6Y",
    }
    cancer_cols = [
        c for c in df.columns
        if c not in ignore and pattern.search(str(c)) and not str(c).endswith("_date")
    ]
    if cancer_cols:
        df["cancer_history"] = (
            df[cancer_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype("int8").max(axis=1)
        )
    else:
        df["cancer_history"] = np.int8(0)
    return df


def sanitize_feature_names(cols):
    out = []
    seen = {}
    for c in cols:
        s = str(c)
        s = s.replace("[", "(").replace("]", ")").replace("<", "lt")
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"[^0-9A-Za-z_(),]+", "_", s)
        if s in seen:
            seen[s] += 1
            s = f"{s}__{seen[s]}"
        else:
            seen[s] = 0
        out.append(s)
    return out


def make_xy(df, ehr_only=False):
    target = "has_lungcancer_3Y"
    y = pd.to_numeric(df[target], errors="raise").astype(int)

    # Match the leakage exclusions used by the paper-cohort PheCode analysis.
    # Outcome, follow-up, index-date, and identifier fields must never be predictors.
    exact_drop = {
        "person_id", "date_of_birth", "race", "ethnicity", "age_at_survey",
        "lifestyle_survey_date", "lungcancer_date", "copd_date",
        "first_ehr_date", "last_ehr_date", "effective_last_ehr_date", "followup_end",
        "followup_days", "followup_years", "fold",
    }
    outcome_drop = {
        c for c in df.columns
        if str(c).lower().startswith("has_lungcancer")
        or str(c).lower() in {"lungcancer", "lung_cancer", "y_true", "outcome"}
    }

    # Exclude diagnosis features that directly name lung cancer. These may encode
    # the target when diagnosis dates or source/standard concept mappings differ.
    lung_target_pattern = re.compile(
        r"(lung|bronchus).*(cancer|carcinoma|malignan|neoplasm)|"
        r"(cancer|carcinoma|malignan|neoplasm).*(lung|bronchus)",
        re.IGNORECASE,
    )
    lung_target_drop = {
        c for c in df.columns if lung_target_pattern.search(str(c))
    }

    drop_cols = exact_drop | outcome_drop | lung_target_drop
    if ehr_only:
        drop_cols |= {
            "duration_of_quitting", "Smoking: Number Of Years",
            "Smoking: Current Daily Cigarette Number", "smoking_status",
        }

    X0 = df.drop(columns=list(drop_cols), errors="ignore").copy()
    cat_cols = list(X0.select_dtypes(include=["object", "string", "category"]).columns)
    X_cat = (
        pd.get_dummies(X0[cat_cols], prefix=cat_cols, dummy_na=True)
        if cat_cols else pd.DataFrame(index=X0.index)
    )
    X_num = X0.drop(columns=cat_cols, errors="ignore")
    for c in X_num.columns:
        X_num[c] = pd.to_numeric(X_num[c], errors="coerce")

    X = pd.concat([X_num, X_cat], axis=1).fillna(0)
    X.columns = sanitize_feature_names(X.columns)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
    X = X.drop(columns=constant_cols, errors="ignore")

    # Fail closed if a future edit allows obvious leakage back into the matrix.
    forbidden = [
        c for c in X.columns
        if "has_lungcancer" in c.lower()
        or c.lower() in {
            "last_ehr_date", "first_ehr_date", "effective_last_ehr_date",
            "followup_days", "followup_years",
        }
        or lung_target_pattern.search(c)
    ]
    if forbidden:
        raise ValueError(f"Target leakage detected in feature matrix: {forbidden[:20]}")

    return X, y



## Cross-Validated Model Runner



In [ ]:
def run_xgb_cv(X, y, fold_ids, label):
    fold_ids = pd.Series(fold_ids, index=X.index).astype(int)
    if not X.index.equals(y.index) or not X.index.equals(fold_ids.index):
        raise ValueError("Features, outcomes, and fold assignments must have aligned indexes.")
    if fold_ids.nunique() != N_SPLITS:
        raise ValueError(f"Expected {N_SPLITS} shared folds; found {fold_ids.nunique()}.")
    oof_pred = np.zeros(len(y), dtype=float)
    aucs, auprcs, briers = [], [], []
    feature_importance_list = []

    print(f"\nRunning {label}")
    print("X shape:", X.shape, "cases:", int(y.sum()), "scale_pos_weight: not used")

    for fold_id in sorted(fold_ids.unique()):
        test_idx = np.flatnonzero(fold_ids.to_numpy() == fold_id)
        train_idx = np.flatnonzero(fold_ids.to_numpy() != fold_id)
        fold = int(fold_id) + 1
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method="hist",
            base_score=0.5,
            n_estimators=100,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
        )
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        oof_pred[test_idx] = proba

        auc = roc_auc_score(y_test, proba)
        auprc = average_precision_score(y_test, proba)
        brier = brier_score_loss(y_test, proba)
        aucs.append(auc)
        auprcs.append(auprc)
        briers.append(brier)

        feature_importance_list.append(
            pd.Series(model.get_booster().get_score(importance_type="gain"), name=f"fold_{fold}")
        )
        print(f"Fold {fold}: AUROC={auc:.4f} AUPRC={auprc:.6f} Brier={brier:.6f} cases={int(y_test.sum())}")

    pooled_auc = roc_auc_score(y, oof_pred)
    pooled_auprc = average_precision_score(y, oof_pred)
    pooled_brier = brier_score_loss(y, oof_pred)
    fi_df = pd.concat(feature_importance_list, axis=1).fillna(0)
    importance_table = fi_df.mean(axis=1).sort_values(ascending=False).reset_index()
    importance_table.columns = ["feature", "mean_gain"]
    importance_table["mean_gain_norm"] = importance_table["mean_gain"] / importance_table["mean_gain"].sum()
    importance_table["model"] = label

    summary = {
        "model": label,
        "n": len(y),
        "cases": int(y.sum()),
        "prevalence": float(y.mean()),
        "auroc_mean": float(np.mean(aucs)),
        "auroc_std": float(np.std(aucs)),
        "auprc_mean": float(np.mean(auprcs)),
        "auprc_std": float(np.std(auprcs)),
        "brier_mean": float(np.mean(briers)),
        "brier_std": float(np.std(briers)),
        "oof_auroc": float(pooled_auc),
        "oof_auprc": float(pooled_auprc),
        "oof_brier": float(pooled_brier),
    }
    pred = pd.DataFrame({
        "person_id": demo_phecode_dates["person_id"].values,
        "y_true": y.values,
        "prob": oof_pred,
        "model": label,
    })
    print("OOF AUROC:", round(pooled_auc, 4), "OOF AUPRC:", round(pooled_auprc, 6), "OOF Brier:", round(pooled_brier, 6))
    display(importance_table.head(20))
    return summary, pred, importance_table



## Run Sensitivity Analyses



In [ ]:
all_summaries = []
all_predictions = []
all_importance = []
all_flips = []

for blackout_days in BLACKOUT_WINDOWS_DAYS:
    sens_df, flips_df = apply_temporal_blackout(demo_phecode_dates, blackout_days=blackout_days)
    sens_df = add_or_refresh_cancer_history(sens_df)
    flips_df["blackout_days"] = blackout_days
    all_flips.append(flips_df)

    print(f"\nBlackout window: {blackout_days} days")
    print("Total feature removals:", int(flips_df["n_removed"].sum()) if len(flips_df) else 0)
    display(flips_df.head(20))

    X, y = make_xy(sens_df, ehr_only=False)
    summary, pred, importance = run_xgb_cv(
        X,
        y,
        fold_ids=sens_df["fold"],
        label=f"Hybrid_PheCode_temporal_{blackout_days}d",
    )
    summary["blackout_days"] = blackout_days
    summary["feature_set"] = "hybrid_phecode"
    all_summaries.append(summary)
    all_predictions.append(pred)
    all_importance.append(importance)

    if RUN_EHR_ONLY:
        X_ehr, y_ehr = make_xy(sens_df, ehr_only=True)
        summary, pred, importance = run_xgb_cv(
            X_ehr,
            y_ehr,
            fold_ids=sens_df["fold"],
            label=f"EHR_only_PheCode_temporal_{blackout_days}d",
        )
        summary["blackout_days"] = blackout_days
        summary["feature_set"] = "ehr_only_phecode"
        all_summaries.append(summary)
        all_predictions.append(pred)
        all_importance.append(importance)

results = pd.DataFrame(all_summaries).sort_values(["feature_set", "blackout_days"]).reset_index(drop=True)
predictions = pd.concat(all_predictions, ignore_index=True)
importance_all = pd.concat(all_importance, ignore_index=True)
flips_all = pd.concat(all_flips, ignore_index=True) if all_flips else pd.DataFrame()
display(results)



## Top 5%, 10%, and 20% Screening Performance

Sensitivity and PPV are calculated from the model's out-of-fold predictions. Participants are ranked from highest to lowest predicted risk, and exactly the requested fraction is selected using `person_id` as a deterministic tie-breaker. The focused table reports only the hybrid PheCode model with a 12-month blackout.


In [ ]:
def top_risk_metrics(prediction_df, fractions=(0.05, 0.10, 0.20)):
    required = {"person_id", "y_true", "prob", "model"}
    missing = required.difference(prediction_df.columns)
    if missing:
        raise ValueError(f"Prediction table is missing columns: {sorted(missing)}")

    rows = []
    for model_name, model_predictions in prediction_df.groupby("model", sort=False):
        if model_predictions["person_id"].duplicated().any():
            raise ValueError(f"{model_name} has duplicate participant predictions.")

        ranked = (
            model_predictions[["person_id", "y_true", "prob"]]
            .dropna()
            .sort_values(["prob", "person_id"], ascending=[False, True])
            .reset_index(drop=True)
        )
        y_true = ranked["y_true"].astype(int)
        if not set(y_true.unique()).issubset({0, 1}):
            raise ValueError(f"{model_name} has a non-binary outcome.")

        total_cases = int(y_true.sum())
        for fraction in fractions:
            n_selected = max(1, int(np.floor(len(ranked) * fraction)))
            selected = ranked.iloc[:n_selected]
            true_positives = int(selected["y_true"].sum())
            false_positives = n_selected - true_positives
            false_negatives = total_cases - true_positives

            rows.append({
                "model": model_name,
                "top_fraction": fraction,
                "top_percent": int(round(fraction * 100)),
                "n": len(ranked),
                "cases": total_cases,
                "selected": n_selected,
                "cases_captured": true_positives,
                "false_positives": false_positives,
                "false_negatives": false_negatives,
                "sensitivity": true_positives / total_cases,
                "ppv": true_positives / n_selected,
            })
    return pd.DataFrame(rows)


top_risk_results = top_risk_metrics(predictions).merge(
    results[["model", "blackout_days", "feature_set"]],
    on="model",
    how="left",
    validate="many_to_one",
)

hybrid_top_risk = (
    top_risk_results.loc[
        top_risk_results["feature_set"].eq("hybrid_phecode")
        & top_risk_results["blackout_days"].eq(365)
    ]
    .sort_values(["blackout_days", "top_fraction"])
    .reset_index(drop=True)
)
hybrid_top_risk["Blackout"] = hybrid_top_risk["blackout_days"].map({
    365: "12-month blackout",
})

paper_rows = []
for blackout_label, group in hybrid_top_risk.groupby("Blackout", sort=False):
    row = {"Model": f"Hybrid PheCode ({blackout_label.lower()})"}
    for _, metric_row in group.iterrows():
        percent = int(metric_row["top_percent"])
        row[f"Top {percent}% Sens./PPV"] = (
            f"{metric_row['sensitivity']:.1%} / {metric_row['ppv']:.1%}"
        )
    paper_rows.append(row)
hybrid_blackout_screening_table = pd.DataFrame(paper_rows).set_index("Model")

print("Hybrid PheCode screening performance: 12-month blackout")
display(hybrid_top_risk[[
    "Blackout", "top_percent", "n", "cases", "selected", "cases_captured",
    "sensitivity", "ppv",
]].style.format({
    "sensitivity": "{:.1%}",
    "ppv": "{:.1%}",
}))

print("Paper-ready sensitivity / PPV table")
display(hybrid_blackout_screening_table)


## Save Outputs



In [ ]:
results_path = datadir / "temporal_phecode_sensitivity_results.csv"
preds_path = datadir / "temporal_phecode_sensitivity_predictions.parquet"
importance_path = datadir / "temporal_phecode_sensitivity_feature_importance.csv"
flips_path = datadir / "temporal_phecode_sensitivity_removed_features.csv"
top_risk_path = datadir / "temporal_phecode_top_risk_metrics.csv"
hybrid_top_risk_path = datadir / "hybrid_phecode_blackout_top_risk_table.csv"

results.to_csv(results_path, index=False)
predictions.to_parquet(preds_path, index=False, compression="snappy")
importance_all.to_csv(importance_path, index=False)
flips_all.to_csv(flips_path, index=False)
top_risk_results.to_csv(top_risk_path, index=False)
hybrid_blackout_screening_table.to_csv(hybrid_top_risk_path)

print("Saved:", results_path)
print("Saved:", preds_path)
print("Saved:", importance_path)
print("Saved:", flips_path)
print("Saved:", top_risk_path)
print("Saved:", hybrid_top_risk_path)



## Camera-Ready Text Template



In [ ]:
print("Use this wording after filling in the exact model values you choose to report:")
print()
print(
    "To assess whether EHR predictors reflected early diagnostic workup rather than antecedent "
    "risk factors, we repeated model training after excluding diagnosis-derived PheCode features "
    "recorded within 12 months before the baseline lifestyle survey date. Model "
    "discrimination was compared with the primary date-filtered model that used diagnoses "
    "recorded on or before the survey date."
)
print()
print("Results table:")
display(results[["model", "blackout_days", "n", "cases", "oof_auroc", "oof_auprc", "oof_brier"]])
